<a href="https://colab.research.google.com/github/collinskiplangatrotich/plp-da-a1-gikomba-stall-analysis/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Dataset provided for Wanjiku's Gikomba Stall
daily_sales = [
    {"day": "Mon", "dresses": 12, "tshirts": 25, "jeans": 8},
    {"day": "Tue", "dresses": 15, "tshirts": 18, "jeans": 10},
    {"day": "Wed", "dresses": 8,  "tshirts": 30, "jeans": 5},
    {"day": "Thu", "dresses": 20, "tshirts": 22, "jeans": 15},
    {"day": "Fri", "dresses": 25, "tshirts": 40, "jeans": 20},
    {"day": "Sat", "dresses": 35, "tshirts": 50, "jeans": 30},
    {"day": "Sun", "dresses": 10, "tshirts": 15, "jeans": 5}
]

prices = {"dresses": 1200, "tshirts": 500, "jeans": 1800}
costs = {"dresses": 700, "tshirts": 250, "jeans": 1000}
inventory = {"dresses": 15, "tshirts": 30, "jeans": 10}

In [2]:
def week_totals(sales_data, item_prices, item_costs):
    """Calculates weekly sales units, revenue, costs, and profit for all items."""
    totals = {"dresses": 0, "tshirts": 0, "jeans": 0}
    total_revenue = 0
    total_cost = 0

    for record in sales_data:
        for item in totals:
            units = record[item]
            totals[item] += units
            total_revenue += units * item_prices[item]
            total_cost += units * item_costs[item]

    total_profit = total_revenue - total_cost

    return {
        "units_sold": totals,
        "total_revenue": total_revenue,
        "total_cost": total_cost,
        "total_profit": total_profit
    }

weekly_summary = week_totals(daily_sales, prices, costs)
print("Weekly Units Sold:", weekly_summary["units_sold"])
print(f"Total Revenue: KES {weekly_summary['total_revenue']:,}")
print(f"Total Profit:  KES {weekly_summary['total_profit']:,}")

Weekly Units Sold: {'dresses': 125, 'tshirts': 200, 'jeans': 93}
Total Revenue: KES 417,400
Total Profit:  KES 186,900


In [3]:

def classify_day(daily_revenue):
    """Classifies a daily revenue amount into performance tiers."""
    if daily_revenue > 50000:
        return "High"
    elif daily_revenue >= 30000:
        return "Average"
    else:
        return "Low"

daily_classifications = []
for record in daily_sales:
    day_rev = (record["dresses"] * prices["dresses"]) + \
              (record["tshirts"] * prices["tshirts"]) + \
              (record["jeans"] * prices["jeans"])
    daily_classifications.append({
        "day": record["day"],
        "revenue": day_rev,
        "tier": classify_day(day_rev)
    })

for day_info in daily_classifications:
    print(f"{day_info['day']}: KES {day_info['revenue']:,} -> {day_info['tier']}")

Mon: KES 41,300 -> Average
Tue: KES 45,000 -> Average
Wed: KES 33,600 -> Average
Thu: KES 62,000 -> High
Fri: KES 86,000 -> High
Sat: KES 121,000 -> High
Sun: KES 28,500 -> Low


In [4]:
def check_reorder_needs(current_inv, units_sold, threshold_days=2):
    """Determines which items require inventory reordering based on remaining days of stock."""
    reorder_summary = {}
    for item, stock in current_inv.items():
        avg_daily_sales = units_sold[item] / 7.0
        days_remaining = stock / avg_daily_sales if avg_daily_sales > 0 else 999

        needs_reorder = days_remaining < threshold_days
        recommended_qty = max(0, int((avg_daily_sales * 7) - stock)) if needs_reorder else 0

        reorder_summary[item] = {
            "current_stock": stock,
            "avg_daily_sales": round(avg_daily_sales, 2),
            "days_remaining": round(days_remaining, 1),
            "reorder_needed": needs_reorder,
            "recommended_order_qty": recommended_qty
        }
    return reorder_summary

reorder_status = check_reorder_needs(inventory, weekly_summary["units_sold"])
for item, details in reorder_status.items():
    print(f"{item.title()}: Stock={details['current_stock']} | Days Left={details['days_remaining']} | Reorder={details['reorder_needed']}")

Dresses: Stock=15 | Days Left=0.8 | Reorder=True
Tshirts: Stock=30 | Days Left=1.1 | Reorder=True
Jeans: Stock=10 | Days Left=0.8 | Reorder=True


In [5]:
best_day = max(daily_classifications, key=lambda x: x["revenue"])
worst_day = min(daily_classifications, key=lambda x: x["revenue"])

print("=" * 62)
print("                WANJIKU GIKOMBA STALL PERFORMANCE REPORT               ")
print("=" * 62)
print(f"{'Item Category':<15} {'Units Sold':<12} {'Revenue (KES)':<16} {'Profit (KES)':<15}")
print("-" * 62)

units = weekly_summary["units_sold"]
for item in prices:
    item_rev = units[item] * prices[item]
    item_prof = units[item] * (prices[item] - costs[item])
    print(f"{item.title():<15} {units[item]:<12} {item_rev:>13,} {item_prof:>13,}")

print("-" * 62)
print(f"{'TOTALS':<15} {sum(units.values()):<12} {weekly_summary['total_revenue']:>13,} {weekly_summary['total_profit']:>13,}")
print("=" * 62)
print(f"Best Sales Day:  {best_day['day']} (KES {best_day['revenue']:,})")
print(f"Worst Sales Day: {worst_day['day']} (KES {worst_day['revenue']:,})")
print("=" * 62)

                WANJIKU GIKOMBA STALL PERFORMANCE REPORT               
Item Category   Units Sold   Revenue (KES)    Profit (KES)   
--------------------------------------------------------------
Dresses         125                150,000        62,500
Tshirts         200                100,000        50,000
Jeans           93                 167,400        74,400
--------------------------------------------------------------
TOTALS          418                417,400       186,900
Best Sales Day:  Sat (KES 121,000)
Worst Sales Day: Sun (KES 28,500)
